# NYC School Success Analysis

This notebook reproduces the full analysis in an educational, step-by-step format.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


## 1. Load the dataset


In [ ]:
df = pd.read_csv("../data/scores.csv")
df.head()


In [ ]:
df.shape
df.info()


## 2. Missing values


In [ ]:
df.isnull().sum().sort_values(ascending=False)


## 3. Keep schools with complete SAT component scores


In [ ]:
sat_columns = [
    "Average Score (SAT Math)",
    "Average Score (SAT Reading)",
    "Average Score (SAT Writing)"
]

schools = df.dropna(subset=sat_columns).copy()
schools.shape


## 4. Feature engineering: Total SAT Score


In [ ]:
schools["Total SAT Score"] = (
    schools["Average Score (SAT Math)"]
    + schools["Average Score (SAT Reading)"]
    + schools["Average Score (SAT Writing)"]
)


## 5. Clean SAT participation


In [ ]:
schools["Percent Tested Numeric"] = (
    schools["Percent Tested"]
    .str.replace("%", "", regex=False)
    .astype(float)
)


In [ ]:
schools[["Total SAT Score", "Percent Tested Numeric", "Student Enrollment"]].describe()


## 6. Borough summary


In [ ]:
borough_summary = schools.groupby("Borough").agg(
    School_Count=("School Name", "count"),
    Average_SAT=("Total SAT Score", "mean"),
    Median_SAT=("Total SAT Score", "median"),
    Average_Percent_Tested=("Percent Tested Numeric", "mean")
)
borough_summary


## 7. Relationship between SAT score and participation


In [ ]:
schools[["Total SAT Score", "Percent Tested Numeric"]].corr()


## 8. Min-Max normalization


In [ ]:
schools["SAT Normalized"] = (
    schools["Total SAT Score"] - schools["Total SAT Score"].min()
) / (
    schools["Total SAT Score"].max() - schools["Total SAT Score"].min()
)

schools["Tested Normalized"] = (
    schools["Percent Tested Numeric"] - schools["Percent Tested Numeric"].min()
) / (
    schools["Percent Tested Numeric"].max() - schools["Percent Tested Numeric"].min()
)


## 9. Composite success score


In [ ]:
schools["Success Score"] = (
    schools["SAT Normalized"] * 0.70
    + schools["Tested Normalized"] * 0.30
)
schools["Success Score 100"] = schools["Success Score"] * 100


## 10. Sensitivity analysis


In [ ]:
for sat_weight, tested_weight, name in [
    (0.50, 0.50, "Score_50_50"),
    (0.80, 0.20, "Score_80_20"),
    (0.90, 0.10, "Score_90_10"),
]:
    schools[name] = (
        schools["SAT Normalized"] * sat_weight
        + schools["Tested Normalized"] * tested_weight
    )

schools["Success Rank"] = schools["Success Score"].rank(ascending=False, method="min")
schools["Rank_50_50"] = schools["Score_50_50"].rank(ascending=False, method="min")
schools["Rank_80_20"] = schools["Score_80_20"].rank(ascending=False, method="min")
schools["Rank_90_10"] = schools["Score_90_10"].rank(ascending=False, method="min")

schools[["Success Rank", "Rank_50_50", "Rank_80_20", "Rank_90_10"]].corr(method="spearman")


## 11. Demographic columns for descriptive analysis


In [ ]:
demographic_columns = [
    "Percent White",
    "Percent Black",
    "Percent Hispanic",
    "Percent Asian"
]

for column in demographic_columns:
    schools[column] = (
        schools[column]
        .str.replace("%", "", regex=False)
        .astype(float)
    )

analysis_columns = [
    "Total SAT Score",
    "Success Score",
    *demographic_columns
]
schools[analysis_columns].corr()


> Demographic correlations are descriptive, school-level associations only. They should not be interpreted as causal or as statements about individual students.


## 12. Final top 10


In [ ]:
top10 = schools.sort_values("Success Score", ascending=False).head(10).copy()
final_top10 = top10[[
    "School Name",
    "Borough",
    "Total SAT Score",
    "Percent Tested Numeric",
    "Success Score 100"
]].copy()

final_top10["Total SAT Score"] = final_top10["Total SAT Score"].astype(int)
final_top10["Success Score 100"] = final_top10["Success Score 100"].round(2)
final_top10 = final_top10.reset_index(drop=True)
final_top10.index = final_top10.index + 1
final_top10


## 13. Export


In [ ]:
final_top10.to_csv("../outputs/top_10_nyc_schools.csv", index_label="Rank")
